In [ ]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [ ]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[2]
sys.path.append(str(repo_path))

In [ ]:
from py.utils import verifyDir,verifyFile

In [ ]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

In [ ]:
QSCORE_PATH=f"{cfg.DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{cfg.DATA_PATH}pp2/images/"

ADE20K_DIR = f"{cfg.DATA_PATH}{cfg.SEG_DATASET}/"
SEGMENT_DIR = f"{cfg.DATA_PATH}pp2/segmentations/{cfg.SEG_DATASET}/{cfg.SEG_MODEL_NAME}/"
GROUP_PATH = f"{cfg.DATA_PATH}pp2/segmentations/{cfg.SEG_DATASET}_group/{cfg.SEG_MODEL_NAME}/"

In [ ]:
verifyDir(GROUP_PATH)

### Loading data

In [ ]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
cities = list(np.sort(data_df["city"].unique()))

### Grouping ADE20k

In [ ]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=cfg.DATA_PATH)
uss.generate_dataset(dataset=f"{cfg.SEG_DATASET}")

In [ ]:
group_df = uss.get_urban_street_categories(by_groups=True)
group_df

In [ ]:
group_df[["group_name", "RGB_color", "hex_color", "classes", "group_class_id"]].to_csv(f"{ADE20K_DIR}/{cfg.SEG_DATASET}_groups_info.csv", sep=";", index=False)

### Mapping classes

In [ ]:
color_dict = group_df.set_index('group_class_id')['RGB_color'].to_dict()
color_dict

In [ ]:
# Step 1: Create a mapping dictionary from class value to class_id
mapping = {
    cls: row['group_class_id']
    for _, row in group_df.iterrows()
    for cls in row['classes']
}

# Step 2: Vectorized replacement using numpy
vectorized_map = np.vectorize(lambda x: mapping.get(x, x))  # defaults to x if not found

### Group Segmentations

In [ ]:
%%time
merge_segment_df = pd.DataFrame()

for idx, current_city in enumerate(cities):
    print(f"{idx+1}: Grouping {current_city}...")
    # Dataset segmentations
    img_masks = np.sort(glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/*.pkl'))
    if len(img_masks)==0:
        continue
    verifyDir(f"{GROUP_PATH}/{current_city}/masks/")
    verifyDir(f"{GROUP_PATH}/{current_city}/ratios/")
    verifyDir(f"{GROUP_PATH}/{current_city}/segmented_images/")
    verifyDir(f"{GROUP_PATH}/{current_city}/segmented_images_overlay/")

    if verifyFile(f"{GROUP_PATH}/{current_city}/segmentations.csv"):
        seg_df = pd.read_csv(f"{GROUP_PATH}/{current_city}/segmentations.csv", sep=";", low_memory=False)
        merge_segment_df = pd.concat([merge_segment_df, seg_df], ignore_index=True)
        merge_segment_df.fillna(0, inplace=True)
        continue

    seg_df = pd.read_csv(f"{SEGMENT_DIR}/{current_city}/segmentations.csv", sep=";", low_memory=False)
    seg_df = uss.process(seg_df, aggregate_classes=True, filter_features=True, keep_disorder=False)
    seg_df.drop(columns=["indoor_object", "outdoor_object", "nature_object"], inplace=True, errors='ignore')
    seg_df.to_csv(f"{GROUP_PATH}/{current_city}/segmentations.csv", sep=";", index=False)
    
    merge_segment_df = pd.concat([merge_segment_df, seg_df], ignore_index=True)
    merge_segment_df.fillna(0, inplace=True)
    
    for cur_mask in tqdm(img_masks):
        current_id = cur_mask.split("/")[-1]
        image_name = current_id.replace(".pkl", "")
        current_image = Image.open(f'{IMAGES_PATH}/{current_city}/{image_name}.JPG'  ).convert("RGB")

        # Class groupes
        ade_mask = joblib.load(cur_mask)
        ade_new_mask = vectorized_map(ade_mask)
        joblib.dump(ade_new_mask, f"{GROUP_PATH}/{current_city}/masks/{current_id}")

        # Group ratio
        unique, counts = np.unique(ade_new_mask, return_counts=True)
        total = ade_new_mask.size
        proportions = {int(k): float(v / total) * 100 for k, v in zip(unique, counts)}
        df = pd.DataFrame(list(proportions.items()), columns=["group_class_id", "ratio"])
        ade_ratio_df = pd.merge(df, group_df[["group_name", "RGB_color", "hex_color", "group_class_id"]].copy(), on="group_class_id", how="left")
        ade_ratio_df.to_csv(f"{GROUP_PATH}/{current_city}/ratios/{image_name}.csv", sep=";", index=False)

        # mask
        ade_new_image = uss.convert_matrix_to_mask(ade_new_mask, color_dict)
        ade_new_image.save(f"{GROUP_PATH}{current_city}/segmented_images/{image_name}.png")

        # overlay
        orig_ade_overlay = Image.blend(current_image, ade_new_image, alpha=0.6)
        orig_ade_overlay.save(f"{GROUP_PATH}/{current_city}/segmented_images_overlay/{image_name}.png")


In [ ]:
merge_segment_df

In [ ]:
merge_segment_df.to_csv(f"{GROUP_PATH}/segmentations.csv", sep=";", index=False)